# 02 - RLHF训练 (PPO算法)

## 学习目标

本notebook深入讲解使用PPO算法进行RLHF训练：

1. **PPO算法原理** - 裁剪目标函数的数学推导
2. **优势函数估计** - GAE算法实现
3. **价值函数网络** - ValueHead架构
4. **训练循环** - 生成-评分-优化完整流程
5. **KL散度控制** - 自适应KL系数

---

## 1. PPO算法理论基础

### 1.1 为什么需要PPO？

传统策略梯度方法存在以下问题：
- 样本效率低
- 训练不稳定
- 超参数敏感

PPO通过**裁剪目标函数**限制策略更新幅度，提高训练稳定性。

### 1.2 PPO裁剪目标

$$L^{CLIP}(\theta) = \mathbb{E}_t[\min(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t)]$$

其中：
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$: 概率比
- $\hat{A}_t$: 优势函数估计
- $\epsilon$: 裁剪参数（通常0.2）

### 1.3 带KL惩罚的完整目标

$$L^{PPO} = \mathbb{E}_t[L^{CLIP}(\theta) - c_1 \cdot \text{KL}(\pi_\theta || \pi_{ref})]$$

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any, Optional

from rlhf import (
    RLHFConfig,
    PPOTrainer,
    ValueHead,
    compute_advantages,
    PPOTrajectory
)

print("环境导入完成！")

## 2. PPO裁剪目标可视化

In [ ]:
def ppo_clip_objective(prob_ratio: np.ndarray, 
                       advantage: np.ndarray, 
                       epsilon: float = 0.2) -> np.ndarray:
    """PPO裁剪目标函数。"""
    # 未裁剪目标
    unclipped = prob_ratio * advantage
    
    # 裁剪后的概率比
    clipped_ratio = np.clip(prob_ratio, 1 - epsilon, 1 + epsilon)
    clipped = clipped_ratio * advantage
    
    # 取最小值
    return np.minimum(unclipped, clipped)

# 可视化不同优势下的目标函数
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 正优势时
prob_ratios = np.linspace(0.5, 1.5, 100)
advantage_pos = 1.0
obj_pos = ppo_clip_objective(prob_ratios, advantage_pos)

axes[0].plot(prob_ratios, prob_ratios * advantage_pos, 'b--', label='未裁剪', alpha=0.5)
axes[0].plot(prob_ratios, obj_pos, 'r-', linewidth=2, label='PPO裁剪目标')
axes[0].axvline(x=0.8, color='gray', linestyle=':', label='裁剪下界')
axes[0].axvline(x=1.2, color='gray', linestyle=':', label='裁剪上界')
axes[0].set_xlabel('概率比 r_t', fontsize=12)
axes[0].set_ylabel('目标函数值', fontsize=12)
axes[0].set_title('正优势 (A_t > 0)', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 负优势时
advantage_neg = -1.0
obj_neg = ppo_clip_objective(prob_ratios, advantage_neg)

axes[1].plot(prob_ratios, prob_ratios * advantage_neg, 'b--', label='未裁剪', alpha=0.5)
axes[1].plot(prob_ratios, obj_neg, 'r-', linewidth=2, label='PPO裁剪目标')
axes[1].axvline(x=0.8, color='gray', linestyle=':', label='裁剪下界')
axes[1].axvline(x=1.2, color='gray', linestyle=':', label='裁剪上界')
axes[1].set_xlabel('概率比 r_t', fontsize=12)
axes[1].set_ylabel('目标函数值', fontsize=12)
axes[1].set_title('负优势 (A_t < 0)', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n关键观察:")
print("- 正优势时：限制策略增长，防止过度优化")
print("- 负优势时：限制策略下降，保持探索")

## 3. GAE优势估计

### 3.1 GAE公式

$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}^V$$

其中 $\delta_t^V = r_t + \gamma V(s_{t+1}) - V(s_t)$ 是TD误差。

In [ ]:
def compute_gae(rewards: np.ndarray,
                values: np.ndarray,
                dones: np.ndarray,
                gamma: float = 0.99,
                lambda_gae: float = 0.95) -> np.ndarray:
    """计算GAE优势估计。
    
    Args:
        rewards: 奖励序列
        values: 价值估计序列
        dones: 终止标志
        gamma: 折扣因子
        lambda_gae: GAE参数
    
    Returns:
        优势函数估计
    """
    advantages = np.zeros_like(rewards)
    gae = 0
    
    # 从后向前计算
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = 0
            next_non_terminal = 1 - dones[t]
        else:
            next_value = values[t + 1]
            next_non_terminal = 1 - dones[t]
        
        # TD误差
        delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
        
        # GAE累积
        gae = delta + gamma * lambda_gae * next_non_terminal * gae
        advantages[t] = gae
    
    return advantages

# 测试GAE计算
rewards = np.array([1.0, 2.0, -1.0, 3.0, 0.5])
values = np.array([0.5, 1.0, 1.5, 2.0, 1.0])
dones = np.array([0, 0, 0, 0, 1])  # 最后一步终止

advantages = compute_gae(rewards, values, dones)

print("GAE优势估计:")
print("="*50)
for i, (r, v, a) in enumerate(zip(rewards, values, advantages)):
    print(f"t={i}: reward={r:5.2f}, value={v:5.2f}, advantage={a:6.3f}")

In [ ]:
# 对比不同lambda值的影响
lambdas = [0.0, 0.5, 0.9, 0.95, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 优势轨迹
for lam in lambdas:
    adv = compute_gae(rewards, values, dones, lambda_gae=lam)
    axes[0].plot(adv, marker='o', label=f'λ={lam}')

axes[0].set_xlabel('时间步', fontsize=12)
axes[0].set_ylabel('优势', fontsize=12)
axes[0].set_title('不同λ值的GAE优势估计', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 方差分析
variances = [np.var(compute_gae(rewards, values, dones, lambda_gae=lam)) for lam in lambdas]
axes[1].bar(range(len(lambdas)), variances, tick_label=[f'λ={lam}' for lam in lambdas])
axes[1].set_ylabel('方差', fontsize=12)
axes[1].set_title('GAE方差 vs λ', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n分析:")
print(f"λ=0: 只使用TD误差，高方差")
print(f"λ=1: 蒙特卡洛估计，低方差但高偏差")
print(f"λ=0.95: 推荐值，平衡方差和偏差")

## 4. PPO训练器配置

In [ ]:
# 创建PPO训练器
config = RLHFConfig(
    # 学习参数
    learning_rate=1e-5,
    ppo_epochs=4,
    batch_size=8,
    
    # PPO特定参数
    clip_epsilon=0.2,      # 裁剪范围
    kl_coef=0.1,           # KL散度系数
    kl_target=0.01,        # 目标KL散度
    
    # GAE参数
    gamma=0.99,            # 折扣因子
    gae_lambda=0.95,       # GAE lambda
)

trainer = PPOTrainer(config)

print(f"PPO配置:")
print(f"  学习率: {config.learning_rate}")
print(f"  PPO轮数: {config.ppo_epochs}")
print(f"  批次大小: {config.batch_size}")
print(f"  裁剪参数: {config.clip_epsilon}")
print(f"  KL系数: {config.kl_coef}")
print(f"  GAE参数: γ={config.gamma}, λ={config.gae_lambda}")

## 5. 生成与评分流程

In [ ]:
# 准备提示
prompts = [
    "解释什么是机器学习",
    "Python的优点是什么",
    "如何提高编程能力",
    "深度学习和机器学习的区别",
    "什么是过拟合现象",
    "如何选择机器学习算法",
    "解释神经网络的工作原理",
    "什么是梯度下降",
]

print(f"准备生成 {len(prompts)} 个回复")

In [ ]:
# 生成回复并计算奖励
batch = trainer.generate_and_score(prompts)

print(f"\n生成结果:")
print(f"""=""*60)
print(f"生成的回复数: {len(batch.responses)}")
print(f"平均奖励: {np.mean(batch.rewards):.4f}")
print(f"奖励标准差: {np.std(batch.rewards):.4f}")
print(f"最高奖励: {np.max(batch.rewards):.4f}")
print(f"最低奖励: {np.min(batch.rewards):.4f}")

# 显示部分样本
print(f"\n前3个样本:")
for i in range(min(3, len(prompts))):
    print(f"\n[{i+1}] {prompts[i]}")
    print(f"回复: {batch.responses[i][:60]}...")
    print(f"奖励: {batch.rewards[i]:.4f}")

## 6. PPO训练循环

In [ ]:
# 完整的训练循环
print("开始PPO训练...")
print("="*60)

training_history = {
    'policy_loss': [],
    'value_loss': [],
    'kl_divergence': [],
    'mean_reward': [],
    'clip_fraction': []
}

num_steps = 10

for step in range(num_steps):
    # 1. 生成并评分
    batch = trainer.generate_and_score(prompts)
    
    # 2. PPO训练
    metrics = trainer.train_step(batch)
    
    # 3. 记录历史
    for key in training_history:
        if key in metrics:
            training_history[key].append(metrics[key])
    
    # 4. 打印进度
    if (step + 1) % 2 == 0:
        print(f"Step {metrics['step']:2d}: "
              f"policy_loss={metrics['policy_loss']:6.3f}, "
              f"value_loss={metrics['value_loss']:6.3f}, "
              f"kl={metrics['kl']:7.5f}, "
              f"reward={metrics['mean_reward']:6.3f}, "
              f"clip_frac={metrics['clip_fraction']:5.2%}")

print("\n训练完成！")

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_to_plot = [
    ('policy_loss', '策略损失'),
    ('value_loss', '价值损失'),
    ('kl_divergence', 'KL散度'),
    ('mean_reward', '平均奖励'),
    ('clip_fraction', '裁剪比例'),
]

for ax, (key, title) in zip(axes[:5], metrics_to_plot):
    ax.plot(training_history[key], 'o-', linewidth=2)
    ax.set_xlabel('Step', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(f'{title}变化', fontsize=12)
    ax.grid(True, alpha=0.3)

# 隐藏多余子图
axes[5].axis('off')

plt.tight_layout()
plt.show()

print("\n关键观察:")
print(f"- 最终平均奖励: {training_history['mean_reward'][-1]:.4f}")
print(f"- 最终KL散度: {training_history['kl_divergence'][-1]:.5f}")
print(f"- 平均裁剪比例: {np.mean(training_history['clip_fraction']):.2%}")

## 7. KL散度自适应控制

In [ ]:
def adaptive_kl_coef(current_kl: float,
                      target_kl: float,
                      current_coef: float,
                      kl_coef_min: float = 0.001,
                      kl_coef_max: float = 0.5) -> float:
    """根据当前KL散度自适应调整KL系数。
    
    实现InstructGPT论文中的自适应策略。
    """
    if current_kl < target_kl / 2:
        # KL太小，增大系数
        return min(current_coef * 1.5, kl_coef_max)
    elif current_kl > target_kl * 2:
        # KL太大，减小系数
        return max(current_coef / 1.5, kl_coef_min)
    else:
        # KL在目标范围内，保持不变
        return current_coef

# 模拟KL系数自适应
kl_values = np.array([0.002, 0.005, 0.008, 0.015, 0.025, 0.006, 0.004])
target_kl = 0.01
kl_coef = 0.1

print(f"初始KL系数: {kl_coef}")
print(f"目标KL: {target_kl}\n")

coef_history = [kl_coef]

for i, kl in enumerate(kl_values, 1):
    kl_coef = adaptive_kl_coef(kl, target_kl, kl_coef)
    coef_history.append(kl_coef)
    print(f"Step {i}: KL={kl:.5f}, 系数={kl_coef:.5f}")

# 可视化
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(kl_values, 'b-o', label='KL散度')
ax1.axhline(y=target_kl, color='gray', linestyle='--', label='目标KL')
ax1.set_ylabel('KL散度', color='b')

ax2.plot(coef_history[1:], 'r-s', label='KL系数')
ax2.set_ylabel('KL系数', color='r')

ax1.set_xlabel('Step')
ax1.set_title('KL系数自适应调整', fontsize=14)
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

plt.show()

## 8. 价值函数网络

In [ ]:
# 创建价值函数头
value_head = ValueHead(hidden_size=256)

# 模拟状态特征
states = np.random.randn(4, 256)  # 4个样本，256维特征

# 计算价值估计
values = value_head(states)

print(f"价值函数输出形状: {values.shape}")
print(f"价值估计: {values.flatten()}")
print(f"平均价值: {values.mean():.4f}")

## 总结

本notebook介绍了PPO-based RLHF的核心内容：

### 关键要点

1. **PPO裁剪目标**: 限制策略更新幅度，提高稳定性
2. **GAE优势估计**: 平衡偏差和方差，lambda=0.95为推荐值
3. **KL散度控制**: 自适应调整KL系数
4. **训练循环**: 生成→评分→优化的完整流程

### PPO超参数建议

| 参数 | 推荐值 | 说明 |
|------|--------|------|
| 学习率 | 1e-5 ~ 5e-5 | 较小的学习率保证稳定 |
| clip_epsilon | 0.2 | 裁剪范围 |
| kl_coef | 0.1 ~ 0.2 | KL散度系数 |
| ppo_epochs | 4 | 每批数据重复使用次数 |
| gamma | 0.99 | 折扣因子 |
| gae_lambda | 0.95 | GAE参数 |

### 下一步

- 学习DPO直接偏好优化 (03_dpo_training.ipynb)